# Q3: Evolution of fires by province
Run the cell below from this notebook’s folder. Each line shows the annual number of fires in one province, from 1998 to 2024.

Colours use the colourblind-friendly Okabe–Ito palette.


In [55]:
from pathlib import Path
import pandas as pd
import altair as alt


def format_chart(chart):
    return (
        chart.configure_title(fontSize=18, subtitleFontSize=12)
        .configure_legend(titleFontSize=14, labelFontSize=12)
        .configure_axis(titleFontSize=14, labelFontSize=12)
    )


fires = pd.read_csv(
    Path("Clean data/CleanData.csv"),
    parse_dates=["data_incendi"],
    dtype={"codi_provincia": "string", "codi_comarca": "string"},
)
fires = fires[fires.data_incendi.dt.year.between(1998, 2024)].copy()
province_names = {"08": "Barcelona", "17": "Girona", "25": "Lleida", "43": "Tarragona"}
province_codes = fires.codi_provincia.str.zfill(2)
fires["provincia"] = province_codes.map(province_names).fillna(fires.provincia)
counts = fires.groupby([fires.data_incendi.dt.year.rename("year"), "provincia"]).size().reset_index(name="fires")
line_chart = alt.Chart(counts).mark_line(point=True).encode(
    x=alt.X("year:Q", title="Year", axis=alt.Axis(format="d"), scale=alt.Scale(zero=False)),
    y=alt.Y("fires:Q", title="Number of fires", scale=alt.Scale(zero=True)),
    color=alt.Color("provincia:N", title="Province", scale=alt.Scale(
        domain=["Barcelona", "Girona", "Lleida", "Tarragona"],
        range=["#56B4E9", "#E69F00", "#009E73", "#CC79A7"])),
    tooltip=["provincia:N", "year:O", "fires:Q"]
).properties(title="Q3: Evolution of fires by province (1998–2024)", height=300, width=700)

province_view = format_chart(line_chart)
province_view


alt.Chart(...)

## Change in fires by comarca
Compare **1998–2002** with **2020–2024** (five years each, inclusive). Color shows `(after − before) / before`: **blue = fewer fires (better), red = more fires (worse)**, white = no change. Hover shows comarca, both counts and percentage change.

Use the `codi_comarca` column in `CleanData.csv` for the **same 43-comarca geography** in both periods, including comarques created after 1998. Exclude records without a comarca code; report their counts below. A zero baseline has no defined percentage change.

Boundaries: [Comarques de Catalunya, ArcGIS](https://services.arcgis.com/fwF8e4aLR6TRWtqM/arcgis/rest/services/comarques_catalunya_etiquetes/FeatureServer/0), saved in `dades/comarques.geojson`. Colors: [ColorBrewer RdBu, colorblind safe](https://colorbrewer2.org/#type=diverging&scheme=RdBu&n=5).


### Boundaries and counts
Project the comarca boundaries before plotting. Use the exported comarca codes to count fires on the same boundaries in both periods.


In [56]:
from math import log1p


from comarca_maps import comarca_areas_hectares, comarca_base_map, load_comarca_boundaries, mercator

boundaries = load_comarca_boundaries(mercator)

comarca = fires.codi_comarca
year = fires.data_incendi.dt.year
before, after = year.between(1998, 2002), year.between(2020, 2024)
codes = [f["properties"]["CODICOMAR"] for f in boundaries["features"]]
comparison = pd.DataFrame({
    "before": comarca[before].value_counts(),
    "after": comarca[after].value_counts(),
})
comparison = comparison.reindex(codes).fillna(0).astype(int).rename_axis("code").reset_index()
comparison["change"] = (comparison.after - comparison.before) / comparison.before.replace(0, float("nan"))
limit = max(comparison.change.abs().max(), 0.01)
rdbu = ["#0571b0", "#92c5de", "#f7f7f7", "#f4a582", "#ca0020"]
print(f"Excluded (unknown comarca): {int((before & comarca.isna()).sum())} before; {int((after & comarca.isna()).sum())} after.")


Excluded (unknown comarca): 4 before; 1 after.


In [57]:
comarca_map = comarca_base_map(boundaries).transform_lookup(
    lookup="properties.CODICOMAR", from_=alt.LookupData(
        comparison,
        "code",
        ["before", "after", "change"],
    )
).encode(
    color=alt.condition(
        "isValid(datum.change)",
        alt.Color(
            "change:Q",
            title="Change (+ = more fires)",
            scale=alt.Scale(domain=[-limit, -limit / 2, 0, limit / 2, limit], range=rdbu),
            legend=alt.Legend(format="+.0%"),
        ),
        alt.value("#bdbdbd"),
    ),
    tooltip=[alt.Tooltip("properties.NOMCOMAR:N", title="Comarca"), alt.Tooltip("before:Q", title="Fires 1998–2002"), alt.Tooltip("after:Q", title="Fires 2020–2024"), alt.Tooltip("change:Q", title="Change", format="+.1%")]
).project(type="identity", reflectY=True).properties(width=650, height=500, title=alt.TitleParams(
    text="Change in fires: 1998–2002 vs 2020–2024",
    subtitle="Blue: fewer fires (better) | Red: more fires (worse)",
)).configure_view(stroke=None)
comarca_view = format_chart(comarca_map)
comarca_view


alt.Chart(...)

## Two equal-time-half comparisons
Each dataset is split exactly halfway between **1 January 1998** and the day after its last recorded fire. Old is before the midpoint; new is on/after it. The dates are printed below each heading.

Compare **fire counts from both sources**, then **total burned hectares from GovEsp only**. Blue = decrease (better), red = increase (worse), white = no change; gray = undefined percentage because the old total is zero. Hover shows both totals and percentage change. Each map has its own symmetric percentage scale. Unmapped municipalities are excluded and counted. The same comarca boundaries are used throughout.


In [58]:
def half_map(data, label):
    data = data[data.data_incendi.ge("1998-01-01")].copy()
    start = pd.Timestamp("1998-01-01")
    end = data.data_incendi.max() + pd.Timedelta(days=1)
    middle = start + (end - start) / 2
    data["code"] = data.codi_comarca
    data["period"] = data.data_incendi.ge(middle).map({False: "before", True: "after"})
    totals = (
        data.groupby(["code", "period"]).amount.sum()
        .unstack(fill_value=0)
        .reindex(index=codes, columns=["before", "after"], fill_value=0)
    )
    totals["change"] = (totals.after - totals.before) / totals.before.replace(0, float("nan"))
    scale = max(totals.change.abs().max(), 0.01)
    print(f"Old: [{start}, {middle}); new: [{middle}, {end}). Excluded unknown comarca: {data.code.isna().sum()}.")
    view = comarca_base_map(boundaries).transform_lookup(
        lookup="properties.CODICOMAR", from_=alt.LookupData(
            totals.reset_index(),
            "code",
            ["before", "after", "change"],
        )
    ).encode(
        color=alt.condition(
            "isValid(datum.change)",
            alt.Color(
                "change:Q",
                title="Change (+ = more)",
                scale=alt.Scale(domain=[-scale, -scale / 2, 0, scale / 2, scale], range=rdbu),
                legend=alt.Legend(format="+.0%"),
            ),
            alt.value("#bdbdbd"),
        ),
        tooltip=[alt.Tooltip("properties.NOMCOMAR:N", title="Comarca"), alt.Tooltip(
            "before:Q",
            title="Old total",
            format=",.2f",
        ), alt.Tooltip(
            "after:Q",
            title="New total",
            format=",.2f",
        ), alt.Tooltip("change:Q", title="Change", format="+.1%")]
    ).project(type="identity", reflectY=True).properties(width=650, height=500, title=alt.TitleParams(
        text=label + ": first half vs second half",
        subtitle="Blue: decrease (better) | Red: increase (worse)",
    )).configure_view(stroke=None)
    return format_chart(view)


### Fire counts: equal halves of 1998–2024


In [59]:
count_map = half_map(fires.assign(amount=1), "Number of fires")
count_map


Old: [1998-01-01 00:00:00, 2011-07-03 00:00:00); new: [2011-07-03 00:00:00, 2025-01-01 00:00:00). Excluded unknown comarca: 14.


### Burned hectares: equal halves of GovEsp coverage only


In [60]:
burned_area_map = half_map(
    fires[fires.font.eq("GovEsp")].assign(amount=lambda d: d.total_hectares),
    "Burned hectares (GovEsp)",
)
burned_area_map


Old: [1998-01-01 00:00:00, 2010-06-27 12:00:00); new: [2010-06-27 12:00:00, 2022-12-22 00:00:00). Excluded unknown comarca: 14.


## Comarca map · Change in annual fire density
Compare **1998–2006** with **2016–2024**, two nine-year periods. Each period’s rate is **(fire count ÷ 9) ÷ comarca hectares × 10,000**. Map colour shows recent minus early rate, in **fires/year per 10,000 ha**. Multiplication by 10,000 only makes the units readable; normalization is by hectares.

**Red = increasing density; blue = decreasing; white = unchanged.** A symmetric scale treats equal increases and decreases equally. For example, +2 means two additional fires per year per 10,000 ha. Tooltips show both rates and comarca area. Normalizing avoids interpreting larger comarques as inherently more fire-prone simply because they contain more land. This is recorded fire density over total land area, not forest-specific risk or burned hectares.

Area source: [Idescat, 2024](https://www.idescat.cat/indicadors/?id=aec&n=15227&t=202400), downloaded to `dades/superficie_comarques_idescat_2024.csv`; published whole km² are converted to hectares (×100). Fixed areas and comarca geography are used for both periods; Val d’Aran matches Idescat’s Aran. Unmapped incidents are excluded. Results depend on period selection and source coverage, including GenCat-only 2023–2024. Earlier charts remain unchanged.

The numeric scale and legend are linear: equal numerical changes have equal spacing. Only the colour palette has mild logarithmic contrast, making moderate changes clearer. Zero stays neutral; positive and negative changes are treated symmetrically. Tooltips retain actual rates.


In [61]:
area_ha = comarca_areas_hectares(boundaries)
early, recent = fires.data_incendi.dt.year.between(1998, 2006), fires.data_incendi.dt.year.between(2016, 2024)
annual_change = pd.DataFrame({
    "early": comarca[early].value_counts(),
    "recent": comarca[recent].value_counts(),
})
annual_change = annual_change.reindex(codes).fillna(0).div(9).rename_axis("code").reset_index()
annual_change["area_ha"] = annual_change.code.map(area_ha)
annual_change[["early", "recent"]] = annual_change[["early", "recent"]].div(
    annual_change.area_ha,
    axis=0,
) * 10000
annual_change["difference"] = annual_change.recent - annual_change.early
extent = max(annual_change.difference.abs().max(), 0.01)
print(f"Excluded unknown comarca: {int((early & comarca.isna()).sum())} early; {int((recent & comarca.isna()).sum())} recent.")
# Linear numeric scale; logarithmic contrast is built into the colour palette.
colour_values = [-extent + 2 * extent * i / 20 for i in range(21)]
colour_range = []
for value in colour_values:
    strength = log1p(abs(value) / (extent / 2)) / log1p(2)
    target = (33, 102, 172) if value < 0 else (178, 24, 43)
    colour_range.append("#" + "".join(f"{round(247 + strength * (channel - 247)):02x}" for channel in target))


Excluded unknown comarca: 10 early; 1 recent.


In [62]:
change_map = comarca_base_map(boundaries).transform_lookup(
    lookup="properties.CODICOMAR", from_=alt.LookupData(
        annual_change,
        "code",
        ["early", "recent", "difference", "area_ha"],
    )
).encode(
    color=alt.Color(
        "difference:Q",
        title="Change: fires/year per 10,000 ha",
        scale=alt.Scale(type="linear", domain=colour_values, range=colour_range),
        legend=alt.Legend(
            values=[-5, -2.5, 0, 2.5, 5],
            labelExpr="datum.value == floor(datum.value) ? format(datum.value, '.0f') : format(datum.value, '.1f')",
        ),
    ),
    tooltip=[alt.Tooltip("properties.NOMCOMAR:N", title="Comarca"), alt.Tooltip(
        "area_ha:Q",
        title="Comarca area (ha)",
        format=",.0f",
    ), alt.Tooltip(
        "early:Q",
        title="1998–2006: fires/year per 10,000 ha",
        format=".2f",
    ), alt.Tooltip(
        "recent:Q",
        title="2016–2024: fires/year per 10,000 ha",
        format=".2f",
    ), alt.Tooltip("difference:Q", title="Change: fires/year per 10,000 ha", format="+.2f")]
).project(type="identity", reflectY=True).properties(width=650, height=500, title=alt.TitleParams(
    text="Change in annual fire density by comarca",
    subtitle="2016–2024 minus 1998–2006 · Per 10,000 ha of total land area",
))
change_view = format_chart(change_map.configure_view(stroke=None))
change_view


alt.Chart(...)

# Final View


In [63]:
final_view = format_chart(alt.vconcat(change_map, line_chart).configure_view(stroke=None))
final_view


alt.VConcatChart(...)

## View explanation


We combined a province line chart with a comarca map to show both annual evolution and geographical differences. Four province lines keep the comparison readable, while chronological position makes trends and unusual years easy to identify. A colourblind-friendly palette distinguishes provinces.

The map compares 1998–2006 with 2016–2024, showing the change in average fires per year per 10,000 hectares. Equal nine-year periods reduce sensitivity to individual years. Dividing by comarca area allows fairer comparisons because larger regions have more land where fires can occur and may show larger absolute changes.

A diverging, colourblind-friendly blue–white–red palette distinguishes decreases from increases, with white representing no change. Red intuitively signals worsening conditions through increasing fire frequency, while blue indicates improvement. Equal increases and decreases receive comparable colour intensity. Tooltips show both periods’ rates, allowing readers to understand the change behind each comarca’s colour.

We first tried percentage changes. These normalize by starting fire counts, rather than land area, and small baselines produced exaggerated variation. Weighting fires by burned hectares was also dominated by large incidents and described severity rather than fire frequency.

The map reveals where frequency changed most, but hides intervening changes. The province lines preserve the annual context.

In [ ]:
from pathlib import Path

export_dir = Path("final_views")
export_dir.mkdir(parents=True, exist_ok=True)
final_view.save(export_dir / "q3.json")